In [ ]:
import requests
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit
import joblib
import os
import json
from datetime import datetime, timedelta
import time
import logging
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

# ─────────────────────────────────────────────
# ★ المتغيرات الأساسية — الفضة
# ─────────────────────────────────────────────
# ★ التنبيه: يُفضل وضع المفتاح في متغيرات البيئة (Environment Variables)
TWELVE_DATA_API_KEY = os.environ.get("TWELVE_DATA_API_KEY", "")
SYMBOL       = "XAG/USD"
SYMBOL_SAFE  = "XAGUSD"
CSV_RAW_FILE = f"{SYMBOL_SAFE}_5min_raw.csv"
MODELS_DIR   = "models/silver"

ATR_MULT_STOP = 1.8
ATR_MULT_TAKE = 2.8
SPREAD_BPS    = 4          # ★ ثابت للفضة

# ─────────────────────────────────────────────
# TIMEFRAME CONFIG
# ★ FIX-3: years مضبوطة + min_candles واقعية لكل فريم
# ─────────────────────────────────────────────
TIMEFRAME_CONFIG = {
    "5min": {
        "interval": "5min", "years": 2,
        "buy_threshold": 0.68, "sell_threshold": 0.32,
        "fold_buy_threshold": 0.62, "fold_sell_threshold": 0.38,
        "max_trades_per_day": 8,   "resample_rule": "5min",
        "larger_tf": "1h",
        "htf_features": ["rsi", "trend_filter", "macd_hist", "market_structure"],
        "minutes_per_candle": 5,   "min_candles": 400
    },
    "15min": {
        "interval": "15min", "years": 3,
        "buy_threshold": 0.68, "sell_threshold": 0.32,
        "fold_buy_threshold": 0.62, "fold_sell_threshold": 0.38,
        "max_trades_per_day": 4,   "resample_rule": "15min",
        "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "atr_pct", "market_structure"],
        "minutes_per_candle": 15,  "min_candles": 300
    },
    "45min": {
        "interval": "45min", "years": 3,
        "buy_threshold": 0.66, "sell_threshold": 0.34,
        "fold_buy_threshold": 0.60, "fold_sell_threshold": 0.40,
        "max_trades_per_day": None, "resample_rule": "45min",
        "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "macd_hist"],
        "minutes_per_candle": 45,  "min_candles": 150
    },
    "1h": {
        "interval": "1h", "years": 4,
        "buy_threshold": 0.65, "sell_threshold": 0.35,
        "fold_buy_threshold": 0.60, "fold_sell_threshold": 0.40,
        "max_trades_per_day": None, "resample_rule": "1h",
        "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist"],
        "minutes_per_candle": 60,  "min_candles": 150
    },
    "2h": {
        "interval": "2h", "years": 4,
        "buy_threshold": 0.65, "sell_threshold": 0.35,
        "fold_buy_threshold": 0.60, "fold_sell_threshold": 0.40,
        "max_trades_per_day": None, "resample_rule": "2h",
        "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "atr_pct"],
        "minutes_per_candle": 120, "min_candles": 100
    },
    "4h": {
        "interval": "4h", "years": 5,
        "buy_threshold": 0.60, "sell_threshold": 0.40,
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.44,
        "max_trades_per_day": None, "resample_rule": "4h",
        "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist"],
        "atr_mult_stop": 2.0, "atr_mult_take": 3.0,     # ★ تكييف وقف الخسارة والهدف
        "minutes_per_candle": 240, "min_candles": 70
    },
    "1d": {
        "interval": "1d", "years": 8,
        "buy_threshold": 0.60, "sell_threshold": 0.40,
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.45,
        "max_trades_per_day": None, "resample_rule": "1B",
        "larger_tf": "1W",
        "htf_features": ["rsi", "trend_filter"],
        "minutes_per_candle": 1440, "min_candles": 70    # ★ FIX
    },
    "1W": {
        "interval": "1week", "years": 12,
        "buy_threshold": 0.57, "sell_threshold": 0.43,
        "fold_buy_threshold": 0.53, "fold_sell_threshold": 0.47,
        "max_trades_per_day": None, "resample_rule": "W-FRI",
        "larger_tf": None, "htf_features": [],
        "minutes_per_candle": 10080, "min_candles": 25   # ★ تقليل لضمان العمل مع داتا أقل
    }
}

# ★ XGB_PARAMS ديناميكية لكل فريم — مطبقة فعلاً في الكود
XGB_PARAMS = {
    "5min": {
        "max_depth": 5, "learning_rate": 0.03, "gamma": 0.30,
        "min_child_weight": 7, "reg_alpha": 0.60, "reg_lambda": 2.0,
        "subsample": 0.65, "colsample_bytree": 0.65
    },
    "15min": {
        "max_depth": 6, "learning_rate": 0.04, "gamma": 0.25,
        "min_child_weight": 6, "reg_alpha": 0.50, "reg_lambda": 1.8,
        "subsample": 0.70, "colsample_bytree": 0.70
    },
    "45min": {
        "max_depth": 6, "learning_rate": 0.04, "gamma": 0.20,
        "min_child_weight": 5, "reg_alpha": 0.40, "reg_lambda": 1.5,
        "subsample": 0.72, "colsample_bytree": 0.72
    },
    "1h": {
        "max_depth": 7, "learning_rate": 0.03, "gamma": 0.15,
        "min_child_weight": 5, "reg_alpha": 0.30, "reg_lambda": 1.3,
        "subsample": 0.75, "colsample_bytree": 0.75
    },
    "2h": {
        "max_depth": 7, "learning_rate": 0.03, "gamma": 0.12,
        "min_child_weight": 4, "reg_alpha": 0.20, "reg_lambda": 1.2,
        "subsample": 0.78, "colsample_bytree": 0.78
    },
    "4h": {
        "max_depth": 8, "learning_rate": 0.015, "gamma": 0.15,
        "min_child_weight": 4, "reg_alpha": 0.20, "reg_lambda": 1.2,
        "subsample": 0.85, "colsample_bytree": 0.85
    },
    "1d": {
        "max_depth": 6, "learning_rate": 0.02, "gamma": 0.08,
        "min_child_weight": 3, "reg_alpha": 0.10, "reg_lambda": 1.0,
        "subsample": 0.82, "colsample_bytree": 0.82
    },
    "1W": {
        "max_depth": 5, "learning_rate": 0.02, "gamma": 0.05,
        "min_child_weight": 2, "reg_alpha": 0.05, "reg_lambda": 0.8,
        "subsample": 0.85, "colsample_bytree": 0.85
    },
    "default": {
        "max_depth": 7, "learning_rate": 0.02, "gamma": 0.10,
        "min_child_weight": 5, "reg_alpha": 0.10, "reg_lambda": 1.0,
        "subsample": 0.80, "colsample_bytree": 0.80
    }
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training_silver.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)


# ══════════════════════════════════════════════════════════════
# DATA FETCHER + MANAGEMENT
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    """
    مسؤول عن جلب بيانات السلاسل الزمنية من Twelve Data API.
    """
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key: str):
        self.api_key = api_key

    def fetch_range(self, symbol=SYMBOL, interval="5min",
                    start_date=None, end_date=None, outputsize=5000):
        """
        جلب بيانات من Twelve Data مع معالجة الأخطاء وإعادة المحاولة التلقائية.
        """
        params = {
            'symbol': symbol, 'interval': interval,
            'apikey': self.api_key, 'format': 'JSON',
            'order': 'ASC', 'outputsize': outputsize
        }
        if start_date:
            params['start_date'] = start_date
        if end_date:
            params['end_date'] = end_date

        logging.info(f"جلب {symbol} ({interval}) ...")

        max_retries = 5
        for attempt in range(max_retries):
            try:
                r = requests.get(self.BASE_URL, params=params, timeout=30)
                r.raise_for_status()
                data = r.json()

                # التحقق من وجود رسالة خطأ من الـ API (مثل تجاوز الحد)
                if 'code' in data and data['code'] == 429:
                    wait_sec = 60
                    logging.warning(f"تم تجاوز حدود الـ API (429). الانتظار {wait_sec} ثانية...")
                    time.sleep(wait_sec)
                    continue

                if 'values' not in data:
                    msg = data.get('message', 'unknown error')
                    # إذا وصلنا لنهاية التاريخ المتاح، نتوقف بهدوء
                    if "No data is available" in msg or "not available" in msg.lower():
                        logging.info(f"ℹ️  وصلنا لنهاية التاريخ المتاح للفريم {interval} (رسالة API: {msg})")
                    else:
                        logging.error(f"❌ خطأ من الـ API للفريم {interval}: {msg}")
                    return pd.DataFrame()

                df = pd.DataFrame(data['values'])
                if df.empty:
                    return df

                if 'volume' not in df.columns:
                    df['volume'] = 0

                for col in ['open', 'high', 'low', 'close', 'volume']:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

                df['datetime'] = pd.to_datetime(df['datetime'])
                df.sort_values('datetime', inplace=True)
                df.reset_index(drop=True, inplace=True)
                return df

            except requests.exceptions.RequestException as e:
                logging.warning(f"محاولة {attempt+1} فشلت: {e}")
                if attempt == max_retries - 1:
                    logging.error("تم استنفاد محاولات الجلب.")
                    break
                # Exponential backoff
                wait_time = (2 ** attempt) + (random.randint(0, 1000) / 1000)
                time.sleep(wait_time)
            except Exception as e:
                logging.error(f"خطأ غير متوقع أثناء الجلب: {e}")
                break

        return pd.DataFrame()


def fetch_full_history(target_candles=100000, interval="5min", max_requests=25):
    """
    جلب تاريخ كامل لرمز معين وفريم معين بحد أقصى للطلبات.
    """
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    all_data = []
    end_date = datetime.utcnow()
    batch_size = 5000

    # تحديد الثواني التقريبية بناءً على الفريم
    multiplier = 60
    if "min" in interval:
        multiplier = int(interval.replace("min", "")) * 60
    elif "h" in interval:
        multiplier = int(interval.replace("h", "")) * 3600
    elif "d" in interval:
        multiplier = 86400
    elif "week" in interval:
        multiplier = 604800

    request_count = 0

    while (len(all_data) * batch_size < target_candles
           and request_count < max_requests):
        request_count += 1
        start_date = end_date - timedelta(seconds=batch_size * multiplier)
        df_batch = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=start_date.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=batch_size
        )
        if df_batch.empty:
            break
        all_data.append(df_batch)
        logging.info(f"تجميع ~{len(all_data) * batch_size} شمعة ({interval}) ...")
        end_date = df_batch['datetime'].min() - timedelta(seconds=1)
        time.sleep(0.6) # احترام حدود الـ API

    if not all_data:
        return pd.DataFrame()
    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    if len(df) > target_candles:
        df = df.iloc[-target_candles:]

    # حفظ الملف باسم يعبر عن الفريم
    csv_name = f"{SYMBOL_SAFE}_{interval}_raw.csv"
    df.to_csv(csv_name, index=False)
    logging.info(f"✅ تم حفظ {len(df)} شمعة في {csv_name}")
    return df


def get_or_update_data(symbol_safe, interval="5min", target_candles=100000,
                       force_refresh=False):
    """
    الحصول على البيانات أو تحديثها لفريم معين.
    """
    csv_path = f"{symbol_safe}_{interval}_raw.csv"
    if os.path.exists(csv_path) and not force_refresh:
        logging.info(f"✅ ملف موجود: {csv_path} — جاري التحديث فقط ...")
        # تحديث بسيط (آخر 1000 شمعة)
        df_old = pd.read_csv(csv_path)
        df_old['datetime'] = pd.to_datetime(df_old['datetime'])
        last_date = df_old['datetime'].max()

        fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
        df_new = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=(last_date + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
            end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=1000
        )
        if df_new.empty:
            return df_old
        df = pd.concat([df_old, df_new], ignore_index=True)
        df.drop_duplicates(subset='datetime', keep='last', inplace=True)
        df.sort_values('datetime', inplace=True)
        df.to_csv(csv_path, index=False)
        return df

    logging.info(f"⬇️  جلب بيانات جديدة ({interval}) ...")
    max_req = 25
    if interval in ["1h", "4h", "1d"]:
        max_req = 60 # جلب تاريخ أعمق للفريمات الكبيرة
    return fetch_full_history(target_candles=target_candles, interval=interval, max_requests=max_req)


def load_csv_for_model(csv_path=CSV_RAW_FILE):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"الملف {csv_path} غير موجود.\n"
            "شغّل train_all_models() أولاً لجلب وحفظ الداتا."
        )
    df = pd.read_csv(csv_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    logging.info(f"✅ تم تحميل {len(df)} شمعة من {csv_path}")
    return df


# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEER — كل المدارس مفعلة ومصحح تماماً
# ══════════════════════════════════════════════════════════════
class FeatureEngineer:
    """
    يحتوي على كافة خوارزميات استخراج الميزات (Features) من بيانات السعر وحجم التداول.
    يدعم التحليل الفني الكلاسيكي، التحليل الرقمي، وسلوك السعر (Price Action).
    """
    def __init__(self):
        self.scaler          = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted   = False

    # ─── 1. Time & Session Features ───────────────────────────
    def add_time_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        إضافة ميزات تعتمد على الوقت وجلسات التداول (لندن، نيويورك، آسيا).
        """
        df = df.copy()
        df['hour']        = df['datetime'].dt.hour
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['month']       = df['datetime'].dt.month
        df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 5)
        df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 5)
        h = df['hour']
        df['is_asia']    = ((h >= 0)  & (h < 9)).astype(int)
        df['is_london']  = ((h >= 8)  & (h < 17)).astype(int)
        df['is_ny']      = ((h >= 13) & (h < 22)).astype(int)
        df['is_overlap'] = ((h >= 13) & (h < 17)).astype(int)
        df['is_friday']  = (df['day_of_week'] == 4).astype(int)
        df['session_momentum'] = (
            df['is_london']   * 2.0
            + df['is_ny']     * 1.5
            + df['is_overlap']* 3.0
            + df['is_asia']   * 0.8
        )
        return df

    # ─── 2. Technical Indicators (adaptive windows) ───────────
    def compute_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        حساب المؤشرات الفنية الأساسية (RSI, MACD, BB, ATR, etc.) بنوافذ زمنية متكيفة.
        """
        df = df.copy()
        c, h, l, o, v = (df['close'], df['high'],
                         df['low'],   df['open'], df['volume'])
        n = len(df)

        # ★ adaptive periods — لا تتجاوز حجم البيانات
        p14  = min(14, max(3, n // 6))
        p20  = min(20, max(4, n // 5))
        p50  = min(50, max(5, n // 4))
        p200 = min(200, max(10, n // 2))

        for span in [9, 21, p50, min(100, max(8, n // 3)), p200]:
            df[f'ema_{span}'] = c.ewm(span=span, adjust=False).mean()
        # نضمن وجود العمدة الثابتة التي نحتاجها لاحقاً
        df['ema_9']   = c.ewm(span=9,   adjust=False).mean()
        df['ema_21']  = c.ewm(span=21,  adjust=False).mean()
        df['ema_50']  = c.ewm(span=p50, adjust=False).mean()
        df['ema_100'] = c.ewm(span=min(100, max(8, n // 3)), adjust=False).mean()
        df['ema_200'] = c.ewm(span=p200, adjust=False).mean()
        df['sma_20']  = c.rolling(p20).mean()
        df['sma_50']  = c.rolling(p50).mean()

        delta = c.diff()
        gain  = delta.clip(lower=0).rolling(p14).mean()
        loss  = (-delta.clip(upper=0)).rolling(p14).mean()
        df['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

        rsi_min = df['rsi'].rolling(p14).min()
        rsi_max = df['rsi'].rolling(p14).max()
        df['stoch_rsi'] = (df['rsi'] - rsi_min) / (rsi_max - rsi_min + 1e-9)

        ema12 = c.ewm(span=min(12, max(3, n // 8)), adjust=False).mean()
        ema26 = c.ewm(span=min(26, max(5, n // 5)), adjust=False).mean()
        macd  = ema12 - ema26
        sig   = macd.ewm(span=min(9, max(3, n // 10)), adjust=False).mean()
        df['macd']        = macd
        df['macd_signal'] = sig
        df['macd_hist']   = macd - sig

        tr = pd.concat([
            h - l,
            (h - c.shift(1)).abs(),
            (l - c.shift(1)).abs()
        ], axis=1).max(axis=1)
        df['atr']     = tr.rolling(p14).mean()
        df['atr_pct'] = df['atr'] / (c + 1e-9)

        bb_mid = c.rolling(p20).mean()
        bb_std = c.rolling(p20).std()
        df['bb_upper'] = bb_mid + 2 * bb_std
        df['bb_lower'] = bb_mid - 2 * bb_std
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (bb_mid + 1e-9)
        df['bb_pct']   = (c - df['bb_lower']) / (
            df['bb_upper'] - df['bb_lower'] + 1e-9
        )

        tp = (h + l + c) / 3
        df['cci'] = (tp - tp.rolling(p20).mean()) / (
            0.015 * tp.rolling(p20).std() + 1e-9
        )

        hh14 = h.rolling(p14).max()
        ll14 = l.rolling(p14).min()
        df['williams_r'] = -100 * (hh14 - c) / (hh14 - ll14 + 1e-9)

        pos_mf = (tp * v).where(tp > tp.shift(1), 0).rolling(p14).sum()
        neg_mf = (tp * v).where(tp < tp.shift(1), 0).rolling(p14).sum()
        df['mfi'] = 100 - (100 / (1 + pos_mf / (neg_mf + 1e-9)))

        plus_dm  = (h - h.shift(1)).clip(lower=0)
        minus_dm = (l.shift(1) - l).clip(lower=0)
        plus_dm  = plus_dm.where(plus_dm  > minus_dm, 0)
        minus_dm = minus_dm.where(minus_dm > plus_dm,  0)
        atr14    = tr.rolling(p14).mean()
        pdi      = 100 * plus_dm.rolling(p14).mean()  / (atr14 + 1e-9)
        mdi      = 100 * minus_dm.rolling(p14).mean() / (atr14 + 1e-9)
        dx       = (pdi - mdi).abs() / (pdi + mdi + 1e-9) * 100
        df['adx']       = dx.rolling(p14).mean()
        df['adx_plus']  = pdi
        df['adx_minus'] = mdi

        s5  = min(5,  max(1, n // 15))
        s10 = min(10, max(2, n // 10))
        s20 = min(20, max(3, n // 7))
        df['momentum_5']  = c - c.shift(s5)
        df['momentum_10'] = c - c.shift(s10)
        df['roc_5']       = c.pct_change(s5)
        df['roc_10']      = c.pct_change(s10)
        df['roc_20']      = c.pct_change(s20)

        body   = (c - o).abs()
        range_ = (h - l).clip(lower=1e-9)
        df['body_ratio'] = body / range_
        df['upper_wick'] = (h - c.clip(lower=o))   / range_
        df['lower_wick'] = (c.clip(upper=o) - l)   / range_
        df['candle_dir'] = np.sign(c - o)

        df['vol_regime'] = c.pct_change().rolling(p20).std()
        df['vol_ratio']  = df['atr'] / (df['atr'].rolling(p50).mean() + 1e-9)

        return df

    # ─── 3. Advanced Volume Analysis ──────────────────────────
    def add_advanced_volume(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        تحليل متقدم لحجم التداول (CVD, VPT, Volume Surge, Imbalance).
        """
        df = df.copy()
        c, v, h, l = df['close'], df['volume'], df['high'], df['low']

        df['volume_delta']   = v * np.sign(c.diff())
        df['cvd']            = df['volume_delta'].cumsum()
        df['cvd_slope']      = df['cvd'].diff(5) / (df['cvd'].abs() + 1e-9)
        df['cvd_divergence'] = np.where(
            df['cvd_slope'] * c.pct_change() < 0, 1, 0
        )

        df['vpt']       = (v * c.pct_change()).cumsum()
        df['vpt_slope'] = df['vpt'].diff(5) / (df['vpt'].abs() + 1e-9)

        vol_ma20 = v.rolling(20).mean()
        vol_ma50 = v.rolling(50).mean()
        df['vol_ratio_20'] = v / (vol_ma20 + 1e-9)
        df['vol_ratio_50'] = v / (vol_ma50 + 1e-9)
        df['volume_surge'] = (v > vol_ma20 * 2).astype(int)

        bull_vol = v.where(c > c.shift(1), 0)
        bear_vol = v.where(c < c.shift(1), 0)
        df['vol_zone_bull'] = bull_vol.rolling(20).mean() / (vol_ma20 + 1e-9)
        df['vol_zone_bear'] = bear_vol.rolling(20).mean() / (vol_ma20 + 1e-9)
        df['vol_imbalance'] = (
            (bull_vol.rolling(10).sum() - bear_vol.rolling(10).sum())
            / (v.rolling(10).sum() + 1e-9)
        )

        price_up = (c.diff() > 0).astype(int)
        vol_up   = (v.diff() > 0).astype(int)
        df['volume_divergence'] = np.where(price_up != vol_up, 1, 0)

        df['eom']     = (
            ((h + l) / 2 - (h.shift(1) + l.shift(1)) / 2)
            / (v / (h - l + 1e-9) + 1e-9)
        )
        df['eom_sma'] = df['eom'].rolling(14).mean()

        return df

    # ─── 4. Price Action Patterns ─────────────────────────────
    def add_price_action(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        اكتشاف نماذج الشموع اليابانية (Pinbar, Engulfing, Dojo, etc.).
        """
        df = df.copy()
        o, h, l, c = df['open'], df['high'], df['low'], df['close']
        body       = (c - o).abs()
        range_     = (h - l + 1e-9)
        upper_wick = h - c.clip(lower=o)
        lower_wick = c.clip(upper=o) - l

        df['pinbar_bull'] = (
            (lower_wick / range_ > 0.6) & (body / range_ < 0.3)
        ).astype(int)
        df['pinbar_bear'] = (
            (upper_wick / range_ > 0.6) & (body / range_ < 0.3)
        ).astype(int)
        df['pinbar'] = (df['pinbar_bull'] | df['pinbar_bear']).astype(int)

        df['engulfing_bull'] = (
            (c.shift(1) < o.shift(1)) & (c > o)
            & (o <= c.shift(1)) & (c >= o.shift(1))
        ).astype(int)
        df['engulfing_bear'] = (
            (c.shift(1) > o.shift(1)) & (c < o)
            & (o >= c.shift(1)) & (c <= o.shift(1))
        ).astype(int)
        df['engulfing'] = (
            df['engulfing_bull'] | df['engulfing_bear']
        ).astype(int)

        df['inside_bar']  = ((h < h.shift(1)) & (l > l.shift(1))).astype(int)
        df['outside_bar'] = ((h > h.shift(1)) & (l < l.shift(1))).astype(int)
        df['doji']        = (body / range_ < 0.1).astype(int)

        df['three_white_soldiers'] = (
            (c > o) & (c.shift(1) > o.shift(1)) & (c.shift(2) > o.shift(2))
            & (c > c.shift(1)) & (c.shift(1) > c.shift(2))
        ).astype(int)
        df['three_black_crows'] = (
            (c < o) & (c.shift(1) < o.shift(1)) & (c.shift(2) < o.shift(2))
            & (c < c.shift(1)) & (c.shift(1) < c.shift(2))
        ).astype(int)

        df['gap_up']      = (o > h.shift(1)).astype(int)
        df['gap_down']    = (o < l.shift(1)).astype(int)
        df['price_accel'] = c.diff().diff()

        return df

    # ─── 5. Trend Filter ──────────────────────────────────────
    def add_trend_filter(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        إضافة ميزات فلترة الاتجاه باستخدام المتوسطات المتحركة (EMA).
        """
        df = df.copy()
        c = df['close']
        df['trend_filter']     = (df['ema_50']  > df['ema_200']).astype(int)
        df['trend_filter_9']   = (df['ema_9']   > df['ema_21']).astype(int)
        df['ema_slope_50']     = df['ema_50'].diff(3)  / (c + 1e-9)
        df['ema_slope_200']    = df['ema_200'].diff(3) / (c + 1e-9)
        df['price_vs_ema50']   = (c > df['ema_50']).astype(int)
        df['price_vs_ema200']  = (c > df['ema_200']).astype(int)
        df['price_vs_ema9']    = (c > df['ema_9']).astype(int)
        df['trend_strength']   = (df['ema_50']  - df['ema_200']) / (c + 1e-9)
        df['trend_strength_9'] = (df['ema_9']   - df['ema_21'])  / (c + 1e-9)
        df['ema_alignment']    = (
            (df['ema_9']   > df['ema_21']).astype(int)
            + (df['ema_21']  > df['ema_50']).astype(int)
            + (df['ema_50']  > df['ema_100']).astype(int)
            + (df['ema_100'] > df['ema_200']).astype(int)
        ) / 4.0
        return df

    # ─── 6. Market Structure ──────────────────────────────────
    def add_market_structure(self, df: pd.DataFrame, window=10) -> pd.DataFrame:
        """
        تحليل هيكل السوق (HH, HL, LH, LL) واكتشاف كسر الهيكل (BOS).
        """
        df = df.copy()
        high, low, c = df['high'], df['low'], df['close']
        rh  = high.rolling(window).max()
        rl  = low.rolling(window).min()
        prh = rh.shift(window)
        prl = rl.shift(window)

        df['hh'] = (rh > prh).astype(int)
        df['hl'] = (rl > prl).astype(int)
        df['lh'] = (rh < prh).astype(int)
        df['ll'] = (rl < prl).astype(int)
        df['market_structure'] = np.where(
            (df['hh'] == 1) & (df['hl'] == 1),  1,
            np.where((df['lh'] == 1) & (df['ll'] == 1), -1, 0)
        )
        df['bos_up']    = (c > rh.shift(1)).astype(int)
        df['bos_down']  = (c < rl.shift(1)).astype(int)
        df['swing_high'] = (
            (high > high.shift(1)) & (high > high.shift(-1))
            & (high > high.shift(2)) & (high > high.shift(-2))
        ).astype(int)
        df['swing_low'] = (
            (low < low.shift(1)) & (low < low.shift(-1))
            & (low < low.shift(2)) & (low < low.shift(-2))
        ).astype(int)
        return df

    # ─── 7. Support & Resistance (adaptive window) ────────────
    def add_support_resistance(self, df: pd.DataFrame, window=20) -> pd.DataFrame:
        """
        تحديد مستويات الدعم والمقاومة واحتساب المسافة إليها.
        """
        df = df.copy()
        c = df['close']
        # ★ FIX: adaptive — لا تتجاوز حجم البيانات
        w  = min(window, max(5, len(df) // 6))
        w50= min(50, max(5, len(df) // 4))

        df['resistance']         = df['high'].rolling(w).max()
        df['support']            = df['low'].rolling(w).min()
        df['dist_to_resistance'] = (df['resistance'] - c) / (c + 1e-9)
        df['dist_to_support']    = (c - df['support'])    / (c + 1e-9)
        df['near_resistance']    = (df['dist_to_resistance'] < 0.002).astype(int)
        df['near_support']       = (df['dist_to_support']    < 0.002).astype(int)
        sr_range      = df['resistance'] - df['support']
        df['sr_dist'] = (c - df['support']) / (sr_range + 1e-9)
        df['resistance_50'] = df['high'].rolling(w50).max()
        df['support_50']    = df['low'].rolling(w50).min()
        df['sr_dist_50']    = (c - df['support_50']) / (
            df['resistance_50'] - df['support_50'] + 1e-9
        )
        return df

    # ─── 8. Volume Profile + VWAP (adaptive window) ───────────
    def add_volume_profile(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        إضافة مؤشر VWAP ونطاقاته، وتحليل OBV.
        """
        df = df.copy()
        c, h, l, v = df['close'], df['high'], df['low'], df['volume']
        tp = (h + l + c) / 3

        # ★ FIX: adaptive window — لا تتجاوز نصف حجم البيانات
        vwap_window = min(144, max(5, len(df) // 4))
        obv_window  = min(20,  max(3, len(df) // 10))

        vol_sum      = v.rolling(vwap_window).sum() + 1e-9
        df['vwap']   = (tp * v).rolling(vwap_window).sum() / vol_sum

        df['dist_to_vwap']  = (c - df['vwap']) / (df['vwap'] + 1e-9)
        df['price_vs_vwap'] = (c > df['vwap']).astype(int)

        vwap_std          = (tp - df['vwap']).rolling(vwap_window).std()
        df['vwap_upper1'] = df['vwap'] + 1.0 * vwap_std
        df['vwap_lower1'] = df['vwap'] - 1.0 * vwap_std
        df['vwap_upper2'] = df['vwap'] + 2.0 * vwap_std
        df['vwap_lower2'] = df['vwap'] - 2.0 * vwap_std
        df['vwap_band_pct'] = (c - df['vwap_lower2']) / (
            df['vwap_upper2'] - df['vwap_lower2'] + 1e-9
        )

        obv_dir          = np.sign(c.diff())
        df['obv']        = (v * obv_dir).cumsum()
        df['obv_sma']    = df['obv'].rolling(obv_window).mean()
        df['obv_slope']  = df['obv'].diff(min(5, len(df) // 8)) / (df['obv'].abs() + 1e-9)
        df['obv_vs_sma'] = (df['obv'] > df['obv_sma']).astype(int)

        return df

    # ─── 9. Digital Indicators (مدرسة رقمية — adaptive) ──────
    def add_digital_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        إضافة مؤشرات "المدرسة الرقمية" (إيشيموكو، كامـا، ZLEMA، الأبعاد الكسرية).
        """
        df = df.copy()
        c, h, l = df['close'], df['high'], df['low']
        n = len(df)

        # ★ adaptive Ichimoku periods
        t9  = min(9,  max(3, n // 8))
        t26 = min(26, max(5, n // 5))
        t52 = min(52, max(8, n // 3))
        tenkan   = (h.rolling(t9).max()  + l.rolling(t9).min())  / 2
        kijun    = (h.rolling(t26).max() + l.rolling(t26).min()) / 2
        shift26  = min(26, max(3, n // 6))
        senkou_a = ((tenkan + kijun) / 2).shift(shift26)
        senkou_b = ((h.rolling(t52).max() + l.rolling(t52).min()) / 2).shift(shift26)
        df['ichi_tenkan']         = tenkan
        df['ichi_kijun']          = kijun
        df['ichi_senkou_a']       = senkou_a
        df['ichi_senkou_b']       = senkou_b
        df['ichi_cloud_bull']     = (senkou_a > senkou_b).astype(int)
        df['ichi_above_cloud']    = ((c > senkou_a) & (c > senkou_b)).astype(int)
        df['ichi_tk_cross']       = (
            (tenkan > kijun) & (tenkan.shift(1) <= kijun.shift(1))
        ).astype(int)
        df['ichi_price_vs_kijun'] = (c > kijun).astype(int)

        # Chande Momentum Oscillator (adaptive)
        cmo_p = min(14, max(3, n // 6))
        d_    = c.diff()
        up_   = d_.clip(lower=0).rolling(cmo_p).sum()
        dn_   = (-d_.clip(upper=0)).rolling(cmo_p).sum()
        df['cmo'] = 100 * (up_ - dn_) / (up_ + dn_ + 1e-9)

        # KAMA (adaptive)
        er_period = min(10, max(3, n // 8))
        fast_sc   = 2 / (2  + 1)
        slow_sc   = 2 / (30 + 1)
        direction  = c.diff(er_period).abs()
        volatility = c.diff().abs().rolling(er_period).sum()
        er         = direction / (volatility + 1e-9)
        sc         = (er * (fast_sc - slow_sc) + slow_sc) ** 2
        kama_arr   = c.to_numpy(dtype=float).copy()
        for i in range(er_period, len(kama_arr)):
            kama_arr[i] = kama_arr[i-1] + sc.iloc[i] * (
                kama_arr[i] - kama_arr[i-1]
            )
        df['kama']          = kama_arr
        df['price_vs_kama'] = (c > df['kama']).astype(int)
        kslope_p = min(3, max(1, n // 12))
        df['kama_slope'] = (
            pd.Series(kama_arr, index=c.index).diff(kslope_p) / (c + 1e-9)
        )

        # Zero-Lag EMA (adaptive)
        zl_p     = min(21, max(5, n // 5))
        lag_     = (zl_p - 1) // 2
        zl_input = c + (c - c.shift(lag_))
        df['zlema']          = zl_input.ewm(span=zl_p, adjust=False).mean()
        df['price_vs_zlema'] = (c > df['zlema']).astype(int)
        zslope_p = min(3, max(1, n // 12))
        df['zlema_slope']    = df['zlema'].diff(zslope_p) / (c + 1e-9)

        # Fractal Dimension (adaptive)
        fd_n = min(20, max(4, n // 5))
        df['fractal_dim'] = (
            h.rolling(fd_n).max() - l.rolling(fd_n).min()
        ) / (df['atr'] * np.sqrt(fd_n) + 1e-9)

        return df

    # ─── 10. HTF Alignment Score ──────────────────────────────
    def add_htf_alignment(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        احتساب نتيجة توافق الفريمات العالية (Score) لتقوية الإشارة.
        """
        df = df.copy()
        idx = df.index

        def _g(col):
            return df[col] if col in df.columns else pd.Series(0.0, index=idx)

        trend     = _g('trend_filter')
        ms        = _g('market_structure').clip(-1, 1) * 0.5
        cvd       = _g('cvd_slope').clip(-1, 1)
        vol_div   = _g('volume_divergence')
        adx_v     = (_g('adx') / 100).clip(0, 1)
        ema_align = _g('ema_alignment')

        df['confirmation_score'] = (
            0.30 * trend
            + 0.20 * ms
            + 0.15 * cvd
            + 0.10 * (1 - vol_div)
            + 0.15 * adx_v
            + 0.10 * ema_align
        )
        df['signal_strength'] = (
            df['confirmation_score']          * 0.4
            + _g('trend_strength').clip(-1, 1) * 0.3
            + _g('vol_imbalance').clip(-1, 1)  * 0.3
        )
        return df

    # ─── MAIN create_features ─────────────────────────────────
    def create_features(self, df: pd.DataFrame, shift=True) -> pd.DataFrame:
        """
        المحرك الرئيسي لإنشاء كافة الميزات ومعالجتها (Scaling + Warming).
        """
        df = self.add_time_features(df)
        df = self.compute_indicators(df)
        df = self.add_advanced_volume(df)       # ★ مفعل
        df = self.add_price_action(df)          # ★ مفعل
        df = self.add_trend_filter(df)
        df = self.add_market_structure(df)
        df = self.add_support_resistance(df)
        df = self.add_volume_profile(df)
        df = self.add_digital_indicators(df)    # ★ مدرسة رقمية
        df = self.add_htf_alignment(df)         # ★ مفعل

        if shift:
            exclude_shift = {'datetime', 'open', 'high', 'low', 'close', 'volume'}
            shift_cols = [col for col in df.columns if col not in exclude_shift]
            df[shift_cols] = df[shift_cols].shift(1)

        # ★ FIX: ffill أولاً قبل الـ warmup — لا نحذف صفوف بسببها
        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        # ★ FIX: warmup adaptive ومخفض لدعم الفريمات الكبيرة
        # max period used = Ichimoku senkou_b shift ≈ n//3 + n//6 ≈ n//2.4
        # لكننا نضع حد أدنى معقول فقط
        warmup = max(10, min(60, len(df) // 10))
        df = df.iloc[warmup:].copy()

        # حذف أعمدة فيها أكثر من 70% NaN (بعد الـ warmup)
        thresh = int(len(df) * 0.3)
        df = df.dropna(axis=1, thresh=thresh)
        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        exclude_final = {
            'datetime', 'open', 'high', 'low', 'close', 'volume',
            'future_close', 'pct_change', 'target'
        }
        self.feature_columns = [col for col in df.columns if col not in exclude_final]
        return df

    def create_target(self, df: pd.DataFrame, horizon=1, threshold=None) -> pd.DataFrame:
        """
        إنشاء الهدف (Target) للتدريب بناءً على تغير السعر المستقبلي.
        """
        df = df.copy()
        df['future_close'] = df['close'].shift(-horizon)
        df['pct_change']   = (df['future_close'] - df['close']) / (df['close'] + 1e-9)
        if threshold is not None:
            df['target'] = np.where(
                df['pct_change'] >  threshold, 2,
                np.where(df['pct_change'] < -threshold, 0, 1)
            )
        else:
            df['target'] = (df['future_close'] > df['close']).astype(int)
        df.dropna(subset=['future_close'], inplace=True)
        df['target'] = df['target'].astype(int)
        return df

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("يجب تدريب الـ scaler أولاً")
        if isinstance(X, pd.DataFrame):
            for col in self.feature_columns:
                if col not in X.columns:
                    X[col] = 0.0
            X = X[self.feature_columns]
        return self.scaler.transform(X)

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)


# ══════════════════════════════════════════════════════════════
# MULTI-TIMEFRAME BUILDER
# ══════════════════════════════════════════════════════════════
class MultiTimeframeBuilder:
    """
    مسؤول عن دمج الميزات من الفريمات الزمنية الأعلى (HTF) في الفريم الحالي.
    """
    def __init__(self):
        self._htf_cache = {}

    def _build_htf_df(self, df_raw: pd.DataFrame, resample_rule: str) -> pd.DataFrame:
        """
        بناء DataFrame للفريم الأعلى ومعالجة ميزاته.
        """
        if resample_rule in self._htf_cache:
            return self._htf_cache[resample_rule]
        df_htf = (
            df_raw.set_index('datetime')
            .resample(resample_rule)
            .agg({'open': 'first', 'high': 'max',
                  'low':  'min',  'close': 'last', 'volume': 'sum'})
            .dropna()
            .reset_index()
        )
        fe_htf   = FeatureEngineer()
        df_htf_f = fe_htf.create_features(df_htf, shift=True)
        df_htf_f = fe_htf.create_target(df_htf_f, horizon=1)
        self._htf_cache[resample_rule] = df_htf_f
        return df_htf_f

    def merge_htf_features(self, df_ltf: pd.DataFrame, df_raw: pd.DataFrame,
                           htf_resample_rule: str, htf_feature_names: list,
                           htf_label: str) -> pd.DataFrame:
        """
        دمج ميزات الـ HTF المحددة في الـ DataFrame الخاص بالفريم الأصغر (LTF).
        """
        if not htf_feature_names or htf_resample_rule is None:
            return df_ltf
        df_htf    = self._build_htf_df(df_raw, htf_resample_rule)
        available = [c for c in htf_feature_names if c in df_htf.columns]
        if not available:
            logging.warning(f"لم يتم إيجاد HTF features لـ {htf_label}")
            return df_ltf
        df_sel = df_htf[['datetime'] + available].copy()
        df_sel.columns = (
            ['datetime'] + [f'htf_{htf_label}_{c}' for c in available]
        )
        df_merged = pd.merge_asof(
            df_ltf.sort_values('datetime'),
            df_sel.sort_values('datetime'),
            on='datetime', direction='backward'
        )
        df_merged = df_merged.ffill().fillna(0)
        return df_merged


# ══════════════════════════════════════════════════════════════
# BACKTESTER
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self, initial_capital=10000, spread_bps=SPREAD_BPS, risk_pct=0.1):
        """
        محاكي صفقات (Backtester) مع إدارة مخاطر قابلة للضبط.
        :param risk_pct: نسبة المخاطرة من رأس المال لكل صفقة (مثلاً 0.1 تعني 10%).
        """
        self.initial_capital = initial_capital
        self.spread_bps      = spread_bps / 10000
        self.slippage        = 0.0001
        self.risk_pct        = risk_pct

    def _exec_price(self, price, is_buy):
        if is_buy:
            return price * (1 + self.spread_bps) * (1 + self.slippage)
        return price * (1 - self.spread_bps) * (1 - self.slippage)

    def run(self, df: pd.DataFrame, model, fe, feature_columns: list,
            timeframe_config: dict,
            atr_mult_stop=ATR_MULT_STOP,
            atr_mult_take=ATR_MULT_TAKE,
            use_fold_thresholds=False) -> dict:
        """
        تشغيل محاكاة التداول على بيانات معينة باستخدام الموديل المدرب.
        :return: قاموس يحتوي على إحصائيات الأداء (Win Rate, Profit Factor, etc.).
        """

        capital   = self.initial_capital
        positions = []
        trades    = []
        equity    = [capital]

        # ★ تحديث: استخدام قيم وقف الخسارة والهدف من الإعدادات إذا وجدت
        a_stop = timeframe_config.get('atr_mult_stop', atr_mult_stop)
        a_take = timeframe_config.get('atr_mult_take', atr_mult_take)

        missing = [c for c in feature_columns + ['close', 'atr']
                   if c not in df.columns]
        if missing:
            logging.error(f"أعمدة مفقودة في Backtester: {missing}")
            return {}

        key_buy  = 'fold_buy_threshold'  if use_fold_thresholds else 'buy_threshold'
        key_sell = 'fold_sell_threshold' if use_fold_thresholds else 'sell_threshold'
        buy_th   = timeframe_config.get(key_buy,  0.60)
        sell_th  = timeframe_config.get(key_sell, 0.40)

        max_daily    = timeframe_config.get('max_trades_per_day')
        daily_counts = {}

        X_scaled  = fe.transform(df[feature_columns].copy())
        all_proba = model.predict_proba(X_scaled)[:, 1]
        closes    = df['close'].values
        atrs      = df['atr'].values
        datetimes = (
            df['datetime'].values if 'datetime' in df.columns
            else [None] * len(df)
        )

        for i in range(len(df)):
            p, atr, prob = closes[i], max(float(atrs[i]), 1e-6), all_proba[i]

            signal = "HOLD"
            if prob > buy_th:
                signal = "BUY"
            elif prob < sell_th:
                signal = "SELL"

            if signal == "BUY" and max_daily and datetimes[i] is not None:
                dk = str(datetimes[i])[:10]
                if daily_counts.get(dk, 0) >= max_daily:
                    signal = "HOLD"

            if signal == "BUY" and not positions:
                ep    = self._exec_price(p, True)
                # استخدام risk_pct للتحكم في حجم الصفقة
                units = capital * self.risk_pct / ep
                positions.append({
                    'entry_price': ep, 'units': units,
                    'stop_loss':   ep - a_stop * atr,
                    'take_profit': ep + a_take * atr
                })
                capital -= units * ep
                if max_daily and datetimes[i] is not None:
                    dk = str(datetimes[i])[:10]
                    daily_counts[dk] = daily_counts.get(dk, 0) + 1

            elif signal == "SELL" and positions:
                pos = positions.pop(0)
                ep  = self._exec_price(p, False)
                pnl = pos['units'] * (ep - pos['entry_price'])
                capital += pos['units'] * ep
                trades.append({
                    'profit':      pnl,
                    'return_pct':  pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                    'exit_reason': 'signal'
                })

            if positions:
                pos = positions[0]
                if p <= pos['stop_loss']:
                    ep  = pos['stop_loss']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({
                        'profit':      pnl,
                        'return_pct':  pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                        'exit_reason': 'stop_loss'
                    })
                    positions.pop(0)
                elif p >= pos['take_profit']:
                    ep  = pos['take_profit']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({
                        'profit':      pnl,
                        'return_pct':  pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                        'exit_reason': 'take_profit'
                    })
                    positions.pop(0)

            mtm = positions[0]['units'] * p if positions else 0
            equity.append(capital + mtm)

        if positions:
            pos = positions.pop(0)
            ep  = self._exec_price(closes[-1], False)
            pnl = pos['units'] * (ep - pos['entry_price'])
            capital += pos['units'] * ep
            trades.append({
                'profit':      pnl,
                'return_pct':  pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                'exit_reason': 'end_of_data'
            })

        if not trades:
            return {}

        eq   = pd.Series(equity)
        wins = [t for t in trades if t['profit'] > 0]
        loss = [t for t in trades if t['profit'] <= 0]
        wr   = len(wins) / len(trades) * 100
        tp   = sum(t['profit'] for t in wins)
        tl   = abs(sum(t['profit'] for t in loss))
        dd   = ((eq - eq.cummax()) / (eq.cummax() + 1e-9) * 100).min()
        aw   = tp / len(wins) if wins else 0
        al   = tl / len(loss) if loss else 0

        return {
            'total_trades':     len(trades),
            'win_rate':         round(wr, 2),
            'profit_factor':    round(tp / tl, 2) if tl > 0 else float('inf'),
            'total_return_pct': round(
                (capital - self.initial_capital) / self.initial_capital * 100, 2
            ),
            'max_drawdown_pct': round(dd, 2),
            'avg_win':          round(aw, 2),
            'avg_loss':         round(al, 2),
            'expectancy':       round(
                (wr / 100 * aw) - ((1 - wr / 100) * al), 2
            ) if al else 0
        }


# ══════════════════════════════════════════════════════════════
# HELPER: تدريب مباشر بدون Walk-Forward (للبيانات الشحيحة)
# ══════════════════════════════════════════════════════════════
def _train_single_fold(X, y, xgb_param, timeframe):
    """يدرّب موديل واحد مباشرةً — fallback آمن للفريمات ذات البيانات القليلة."""
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    spw = neg / pos if pos > 0 else 1.0

    fe_fold = FeatureEngineer()
    fe_fold.fit_scaler(X)
    X_s = fe_fold.transform(X)

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=400,
        scale_pos_weight=spw,
        random_state=42,
        tree_method='hist',
        eval_metric='logloss',
        verbosity=0,
        **xgb_param
    )
    model.fit(X_s, y, verbose=False)
    logging.info(
        f"[{timeframe}] ✅ Single-fold training — {len(X)} عينة"
    )
    return model, fe_fold


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD TRAINING
# ══════════════════════════════════════════════════════════════
def train_model_with_walk_forward(df_raw, timeframe, n_splits=5,
                                   mtf_builder=None):
    config        = TIMEFRAME_CONFIG[timeframe]
    resample_rule = config['resample_rule']
    years         = config['years']
    min_candles   = config.get('min_candles', 100)
    larger_tf     = config.get('larger_tf')
    htf_features  = config.get('htf_features', [])

    logging.info("=" * 60)
    logging.info(f"Walk-Forward [SILVER / XAG] — فريم {timeframe}")
    logging.info("=" * 60)

    # Resample
    df_tf = (
        df_raw.set_index('datetime')
        .resample(resample_rule)
        .agg({'open': 'first', 'high': 'max',
              'low':  'min',  'close': 'last', 'volume': 'sum'})
        .dropna()
        .reset_index()
    )
    if 'datetime' not in df_tf.columns:
        df_tf.rename(columns={'index': 'datetime'}, inplace=True)

    if years:
        cutoff = df_tf['datetime'].max() - timedelta(days=365 * years)
        df_tf  = df_tf[df_tf['datetime'] >= cutoff].copy()

    logging.info(f"شموع بعد القطع ({years}y): {len(df_tf)}")

    if len(df_tf) < min_candles:
        logging.error(
            f"❌ بيانات غير كافية للفريم {timeframe}: "
            f"{len(df_tf)} < {min_candles}"
        )
        return None

    # Features & Target
    fe        = FeatureEngineer()
    df_feat   = fe.create_features(df_tf, shift=True)
    df_target = fe.create_target(df_feat, horizon=1)

    if df_target.empty or len(df_target) < min_candles:
        logging.error(
            f"❌ بيانات غير كافية بعد إنشاء features للفريم {timeframe}: "
            f"{len(df_target)} صف (المطلوب {min_candles})"
        )
        return None

    # HTF merge
    if mtf_builder and larger_tf and htf_features:
        htf_cfg = TIMEFRAME_CONFIG.get(larger_tf)
        if htf_cfg:
            logging.info(f"إضافة HTF features من {larger_tf} ...")
            df_target = mtf_builder.merge_htf_features(
                df_ltf=df_target,
                df_raw=df_raw,
                htf_resample_rule=htf_cfg['resample_rule'],
                htf_feature_names=htf_features,
                htf_label=larger_tf
            )

    exclude_final = {
        'datetime', 'open', 'high', 'low', 'close', 'volume',
        'future_close', 'pct_change', 'target'
    }
    fe.feature_columns = [c for c in df_target.columns if c not in exclude_final]
    logging.info(f"إجمالي الـ features: {len(fe.feature_columns)}")

    X = df_target[fe.feature_columns]
    y = df_target['target']

    xgb_param = XGB_PARAMS.get(timeframe, XGB_PARAMS["default"])

    # حسب حجم البيانات نحدد الحد الأدنى لكل فولد بشكل تكيفي
    if len(X) < 100:
        # للبيانات الصغيرة: نكتفي بمتطلبات أقل لضمان عمل الـ Walk-Forward
        min_train = max(15, len(X) // 4)
        min_test  = max(5,  len(X) // 10)
        n_splits  = min(n_splits, 3)
    else:
        # للبيانات الكبيرة: نلتزم بمعايير صارمة
        min_train = max(30, len(X) // (n_splits + 1))
        min_test  = max(8,  len(X) // (n_splits * 3))

    effective_splits = min(n_splits, max(2, len(X) // (min_train + min_test)))
    tscv = TimeSeriesSplit(n_splits=effective_splits)

    fold_metrics = []
    best_model   = None
    best_fe_fold = None
    best_score   = -np.inf
    # ★ FIX: نتبع آخر نموذج مدرَّب بشكل آمن (لمنع UnboundLocalError)
    last_model   = None
    last_fe_fold = None

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        if len(X_tr) < min_train or len(X_te) < min_test:
            logging.warning(
                f"Fold {fold+1}/{effective_splits}: "
                f"train={len(X_tr)} test={len(X_te)} — تخطي"
            )
            continue

        logging.info(
            f"── Fold {fold+1}/{effective_splits} "
            f"(train={len(X_tr)}, test={len(X_te)})"
        )

        neg = (y_tr == 0).sum()
        pos = (y_tr == 1).sum()
        spw = neg / pos if pos > 0 else 1.0

        fe_fold = FeatureEngineer()
        fe_fold.fit_scaler(X_tr)
        X_tr_s = fe_fold.transform(X_tr)
        X_te_s = fe_fold.transform(X_te)

        model = xgb.XGBClassifier(
            objective='binary:logistic',
            n_estimators=800,
            early_stopping_rounds=60,
            scale_pos_weight=spw,
            random_state=42,
            tree_method='hist',
            eval_metric='logloss',
            verbosity=0,
            **xgb_param    # ★ ديناميكية مطبقة
        )
        model.fit(
            X_tr_s, y_tr,
            eval_set=[(X_te_s, y_te)],
            verbose=False
        )

        # ★ FIX: حفظ آخر نموذج دائماً حتى لو لم يكن الأفضل
        last_model   = model
        last_fe_fold = fe_fold

        y_pred    = model.predict(X_te_s)
        acc       = accuracy_score(y_te, y_pred)
        precision = precision_score(y_te, y_pred, zero_division=0)
        recall    = recall_score(y_te, y_pred, zero_division=0)

        df_test      = df_target.iloc[te_idx].copy()
        backtester   = Backtester()
        metrics_test = backtester.run(
            df_test, model, fe_fold,
            fe_fold.feature_columns, config,
            use_fold_thresholds=True
        )

        logging.info(
            f"  Fold {fold+1} — "
            f"Acc:{acc:.2%}  Prec:{precision:.2%}  Rec:{recall:.2%}"
        )
        if metrics_test:
            logging.info(
                f"  BT — WR:{metrics_test['win_rate']}%  "
                f"PF:{metrics_test['profit_factor']}  "
                f"Ret:{metrics_test['total_return_pct']}%  "
                f"Trades:{metrics_test['total_trades']}"
            )
        else:
            logging.info("  BT — لا صفقات في هذا الفولد")

        fold_metrics.append({
            'fold': fold + 1, 'accuracy': acc,
            'precision': precision, 'recall': recall,
            'backtest': metrics_test
        })

        # ★ FIX: Score مركب شامل — لا يعتمد على عدد الصفقات فقط
        if metrics_test:
            score = (
                metrics_test.get('profit_factor', 0) * 0.55
                + metrics_test.get('win_rate',    0) * 0.25
                + metrics_test.get('total_trades',0) * 0.15
                + acc                                * 0.05
            )
        else:
            # لو مفيش backtest: accuracy وحدها كـ fallback
            score = acc * 10

        if score > best_score:
            best_score   = score
            best_model   = model
            best_fe_fold = fe_fold

    # ★ FIX: Fallback آمن 100% — لا UnboundLocalError أبداً
    if best_model is None:
        logging.warning(
            f"⚠️  [{timeframe}] لم يتم اختيار fold جيد → "
            f"Single Fold Training مباشرة"
        )
        best_model, best_fe_fold = _train_single_fold(X, y, xgb_param, timeframe)

    # Final Model على كامل البيانات
    neg_f = (y == 0).sum()
    pos_f = (y == 1).sum()
    n_est = 800
    if hasattr(best_model, 'best_iteration') and best_model.best_iteration:
        n_est = max(100, best_model.best_iteration + 1)

    fe_full = FeatureEngineer()
    fe_full.fit_scaler(X)
    X_full_s = fe_full.transform(X)

    final_model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=n_est,
        scale_pos_weight=neg_f / pos_f if pos_f else 1,
        random_state=42,
        tree_method='hist',
        eval_metric='logloss',
        verbosity=0,
        **xgb_param
    )
    final_model.fit(X_full_s, y)

    # Save
    os.makedirs(MODELS_DIR, exist_ok=True)
    model_path = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}.pkl"
    joblib.dump({
        'model':           final_model,
        'scaler':          fe_full.scaler,
        'feature_columns': fe_full.feature_columns,
        'config':          config
    }, model_path)

    # Holdout Backtest (آخر 20%)
    holdout_start = int(len(df_target) * 0.8)
    df_holdout    = df_target.iloc[holdout_start:].copy()
    full_metrics  = {}

    if len(df_holdout) >= 10:
        bt_holdout   = Backtester()
        full_metrics = bt_holdout.run(
            df_holdout, final_model, fe_full,
            fe_full.feature_columns, config,
            use_fold_thresholds=False
        )
        if full_metrics:
            logging.info(
                f"✅ Holdout — WR:{full_metrics['win_rate']}%  "
                f"PF:{full_metrics['profit_factor']}  "
                f"Ret:{full_metrics['total_return_pct']}%  "
                f"Trades:{full_metrics['total_trades']}"
            )
        else:
            logging.info(
                f"ℹ️  [{timeframe}] لا صفقات في Holdout "
                f"(thresholds مرتفعة — طبيعي للفريمات الكبيرة)"
            )
    else:
        logging.warning(
            f"⚠️  [{timeframe}] holdout صغير ({len(df_holdout)} صف) — تخطي"
        )

    # Save Metrics JSON
    metrics_file = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}_metrics.json"
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump({
            'symbol':                      SYMBOL,
            'timeframe':                   timeframe,
            'n_features':                  len(fe_full.feature_columns),
            'fold_results':                fold_metrics,
            'holdout_backtest_last_20pct': full_metrics,
            'config':                      config
        }, f, indent=2, default=str, ensure_ascii=False)

    logging.info(f"💾 تم حفظ النموذج: {model_path}")
    return final_model, fe_full


# ══════════════════════════════════════════════════════════════
# TRAIN ALL MODELS
# ══════════════════════════════════════════════════════════════
def train_all_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("★ بدء نظام Model Factory — Silver (XAG/USD) Optimization ★")
    logging.info("=" * 70)

    # جلب داتا الـ 5 دقائق (للفريمات الصغيرة)
    df_5min = get_or_update_data(SYMBOL_SAFE, interval="5min", force_refresh=force_refresh_data)

    # جلب داتا الـ ساعه (للفريمات المتوسطة والكبيرة لضمان عمق تاريخي)
    df_1h = get_or_update_data(SYMBOL_SAFE, interval="1h", force_refresh=force_refresh_data)

    mtf_builder = MultiTimeframeBuilder()
    results     = {}

    for tf in TIMEFRAME_CONFIG:
        logging.info("-" * 50)
        try:
            # اختيار الداتا المصدرية الأنسب للفريم
            # الفريمات الكبيرة تحتاج داتا "ساعة" ليكون لديها تاريخ كافٍ (سنوات)
            if tf in ["1h", "2h", "4h", "1d", "1W"]:
                source_df = df_1h
            else:
                source_df = df_5min

            if source_df is None or source_df.empty:
                results[tf] = "❌ فشل الحصول على البيانات"
                continue

            res = train_model_with_walk_forward(
                source_df, tf, n_splits=5, mtf_builder=mtf_builder
            )
            results[tf] = "✅ نجح" if res else "❌ فشل (بيانات غير كافية)"
        except Exception as e:
            logging.error(
                f"❌ خطأ في تدريب الفريم {tf}: {e}", exc_info=True
            )
            results[tf] = f"❌ خطأ: {e}"

    logging.info("=" * 70)
    logging.info("ملخص التدريب الفائق — Silver (XAG/USD):")
    for tf, st in results.items():
        logging.info(f"   {tf:6s}: {st}")
    logging.info("=" * 70)
    logging.info(f"📁 الموديلات: {MODELS_DIR}/")
    logging.info(f"📊 الداتا:    {CSV_RAW_FILE}")


# ══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    train_all_models(force_refresh_data=True)

In [10]:
# ============================================================
# download_models


import os
import shutil
from google.colab import files

MODELS_DIR = "models/silver"
ZIP_NAME   = "silver_models.zip"

# ── تحقق من وجود الفولدر ──
if not os.path.exists(MODELS_DIR):
    print(f"❌ الفولدر '{MODELS_DIR}' مش موجود!")
    print("تأكد إن silver_v35.py اشتغل وحفظ الملفات أولاً.")
else:
    # ── اعرض الملفات الموجودة ──
    all_files = os.listdir(MODELS_DIR)
    pkl_files  = [f for f in all_files if f.endswith('.pkl')]
    json_files = [f for f in all_files if f.endswith('.json')]

    print(f"📁 محتوى '{MODELS_DIR}':")
    print(f"   .pkl  (موديلات): {len(pkl_files)}")
    print(f"   .json (مقاييس):  {len(json_files)}")
    print()
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<45} {size/1024:.1f} KB")

    # ── اضغط الفولدر ──
    print(f"\n📦 جاري ضغط الفولدر ...")
    shutil.make_archive(
        ZIP_NAME.replace('.zip', ''),  # اسم الملف
        'zip',                          # نوع الضغط
        '.',                            # المجلد الأساسي
        MODELS_DIR                      # الفولدر المراد ضغطه
    )

    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"✅ تم إنشاء: {ZIP_NAME} ({size_mb:.1f} MB)")

    # ── نزّل الملف ──
    print(f"⬇️  جاري التحميل ...")
    files.download(ZIP_NAME)
    print(f"✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.")

📁 محتوى 'models/silver':
   .pkl  (موديلات): 8
   .json (مقاييس):  8

   XAGUSD_15min.pkl                              858.1 KB
   XAGUSD_15min_metrics.json                     2.3 KB
   XAGUSD_1W.pkl                                 155.1 KB
   XAGUSD_1W_metrics.json                        1.7 KB
   XAGUSD_1d.pkl                                 238.9 KB
   XAGUSD_1d_metrics.json                        1.3 KB
   XAGUSD_1h.pkl                                 548.6 KB
   XAGUSD_1h_metrics.json                        1.4 KB
   XAGUSD_2h.pkl                                 473.1 KB
   XAGUSD_2h_metrics.json                        1.6 KB
   XAGUSD_45min.pkl                              568.0 KB
   XAGUSD_45min_metrics.json                     2.1 KB
   XAGUSD_4h.pkl                                 588.4 KB
   XAGUSD_4h_metrics.json                        1.7 KB
   XAGUSD_5min.pkl                               262.8 KB
   XAGUSD_5min_metrics.json                      2.3 KB

📦 جاري ضغط الفولد

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.
